# Resturant API

## database.py

In [ ]:
# database.py:

from motor.motor_asyncio import AsyncIOMotorClient
from pymongo import ASCENDING
import os
from dotenv import load_dotenv

load_dotenv()                            # Load .env file

MONGO_URL = os.getenv("MONGO_URI")       # fetch connection url from environment

client = AsyncIOMotorClient(MONGO_URL)   # create connection to mongoDB server to perform opeartion

db = client["ecommerce_db"]              # database 

product_collection = db.products         # collection for storing product details
order_collection = db.orders             # collection for storing order details 


**What is `motor`?**
- motor is the asynchronous Python driver for MongoDB.
- It's built on top of `PyMongo` but supports `async` and `await` syntax, making it ideal for async frameworks like FastAPI, Starlette, and Tornado.

**AsyncIOMotorClient**
- This is the async equivalent of `pymongo.MongoClient`.
- It creates a client connection to your MongoDB server, allowing async operations like `.find()`, `.insert_one()`, etc.
- Non-blocking I/O: other FastAPI routes aren't blocked while DB operations are happening.

## schemas.py

In [ ]:
# schemas.py:

from typing import List, Optional
from pydantic import BaseModel, Field, field_validator, model_validator
from bson import ObjectId


class SizeItem(BaseModel): # used by ProductCreate model for sizes
    
    size: str            # pydantic model fields
    quantity: int        # pydantic model fields
    
    @field_validator("quantity") # validating quantity for negative value
    @classmethod
    def quantity_non_negative(cls, value):
        if value < 0:
            raise ValueError("Quantity must be >= 0")
        return value

- Attributes `size`, `quantity` are not traditional `class variables` or `instance variables` in the normal Python sense. Instead, in Pydantic (and dataclasses too), these are `Pydantic model fields`.

In [ ]:
# POST products/
class ProductCreate(BaseModel): # used in POST products/ for validaing data from user for creating new product
    name: str
    price: float
    sizes: List[SizeItem]
    
    @field_validator("price")
    @classmethod
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("Price must be greater than 0")
        return v
    
    @model_validator(mode="after") # validating model for duplicate size
    # model-level validator that runs after all field validations
    def validate_unique_sizes(self): # Here 'self' refers to the ProductCreate instance.   
        seen_sizes = set()
        for item in self.sizes:    # self.sizes is already validated as List[SizeItem]
            if item.size in seen_sizes:
                raise ValueError(f"Duplicate size found: {item.size}")
            seen_sizes.add(item.size)
        
        return self # ✅ always return self in mode="after"

        # Rule of thumb:
        #     mode="after" → Always return self (or a modified instance).
        #     mode="before" → Return a dict of data instead, because at that stage the model isn’t built yet.



class ProductResponse(BaseModel): # used in POST products/ as response model for validating returned response
    id: str

- `ProductCreate` is a `Pydantic model` (inherits from `BaseModel`) used for data validation when creating a new product.

**Why use `@model_validator(mode="after")` here?**

Because we're validating the relationship between items in a list, not a single field. Field-level validators (`@field_validator`) work on one field only. This validator checks the entire list for duplicates after field types have already been validated.

In [ ]:
# GET products/
class ProductSummary(BaseModel): # used in GET products/ requesto validate product details to be shown in reponse
    id: str
    name: str
    price: float

class ProductListResponse(BaseModel): # used in GET products/ to validate response data
    data: List[ProductSummary]
    page: dict


In [ ]:
# POST orders/
class OrderItemInput(BaseModel):
    productId: str
    qty: int

    @field_validator("qty")
    @classmethod
    def quantity_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("Quantity must be greater than 0")
        return v


class OrderCreate(BaseModel): # used in POST orders/ request for validating data coming from user 
    userId: str
    items: List[OrderItemInput]


class OrderResponse(BaseModel): # used in POST orders/ for validating response
    id: str


In [ ]:
# GET orders/<user_id>
class ProductDetails(BaseModel):
    name: str
    id: str

class OrderItemOut(BaseModel):
    productDetails: ProductDetails
    qty: int

class OrderOut(BaseModel):
    id: str
    items: List[OrderItemOut]
    total: float

class PaginationInfo(BaseModel):
    next: int
    limit: int
    previous: int

class OrderListResponse(BaseModel): # used in GET orders/<user_id> for validating response
    data: List[OrderOut]
    page: PaginationInfo 

## main.py

In [ ]:
# main.py:

from fastapi import FastAPI, Query, HTTPException, Path
from typing import List, Optional
from bson import ObjectId

from database import product_collection, order_collection
from schemas import ProductCreate, ProductResponse, ProductListResponse, ProductSummary, OrderCreate, OrderResponse, OrderListResponse

app = FastAPI()

# Create Product API
@app.post("/products", response_model=ProductResponse, status_code=201)
async def create_product(product: ProductCreate):

    print(product)        # name='Cap' price=199.0 sizes=[SizeItem(size='regular', quantity=200)]
    print(type(product))  # <class 'schemas.ProductCreate'>
    
    product_dict = product.model_dump() # convert Pydantic object product into dictionary.
    result = await product_collection.insert_one(product_dict) 

    print(result)          #output:  InsertOneResult(ObjectId('688fa09185c7b3813efc1b43'), acknowledged=True)
    print(type(result))    #output:  <class 'pymongo.results.InsertOneResult'>
    
    return {"id": str(result.inserted_id)} # # should only includes what's defined in ProductResponse


1. When someone sends a `POST` request with a product, this function `create_product()` will be called. FastAPI will automatically parse and validate the request body as a `ProductCreate` model (defined in `schemas.py`).
2. `create_product()` expects a request body matching the schema `ProductCreate` (which includes `name`, `price`, and `sizes`).
    - So `product` is validated by `ProductCreate` schema and return a Pydantic object.
3. `model_dump()` converts the validated Pydantic object product into a plain Python dictionary. Because `motor` (your async MongoDB driver) expects regular dictionaries to insert into MongoDB. This format is needed for inserting into MongoDB.
4. `insert_one()` is an async MongoDB method that inserts a document and returns an object type `InsertOneResult` that includes `inserted_id`.
5. `inserted_id` is then converted into `str` and then `response_model=ProductResponse` → The output/response will follow the `ProductResponse` schema.

**Why are we doing `str(result.inserted_id)`?**

MongoDB stores data in BSON(Binary JSON) format. When documents are stored in MongoDB, their IDs (`_id`) are usually of type `ObjectId` and is unique identifier.

Example: `ObjectId("64c77f44fe45d8a1e7f7a9b1")`

It's not directly serializable to JSON, which means you can't return it in an API response as-is — it must be converted to a string. However, FastAPI (or any JSON-based API) cannot serialize `ObjectId` directly, so you need to transform it into a plain string.

**JSON only allows the following types:**

✅ Allowed: `string`, `number`, `boolean`, `null`, `array`, `object (dictionary)`

❌ Not allowed: `Python datetime objects`, `MongoDB ObjectId`, `Python custom classes`

| Term              | Meaning                                                       |
| ----------------- | ------------------------------------------------------------- |
| Serializable      | Can be converted to a format like JSON                        |
| JSON-serializable | Can be encoded into JSON (string, number, etc.)               |
| ObjectId          | Special MongoDB type, **not** serializable into JSON directly |
| 

In [ ]:
# List Products API
@app.get("/products", response_model=ProductListResponse)
async def list_products(
    name: Optional[str] = None, # filter product by name
    size: Optional[str] = None, # filter product by size
    limit: int = 10,
    offset: int = 0
):
    query = {}
    if name:
        query["name"] = {"$regex": name, "$options": "i"} # If a name is provided, filter products whose name contains the string (case-insensitive)
    if size:
        query["sizes.size"] = size  # search inside embedded sizes
    
    # /products?name=sneakers
    print(query) # {'name': {'$regex': 'sneakers', '$options': 'i'}}
    
    cursor = product_collection.find(query).sort("_id").skip(offset).limit(limit)
    print(cursor) # AsyncIOMotorCursor(<pymongo.synchronous.cursor.Cursor object at 0x000001C05C9BB8E0>)
    print(type(cursor)) # <class 'motor.motor_asyncio.AsyncIOMotorCursor'>
    
    products = [] # store list of Pydantic model instances (ProductSummary)
    
    async for doc in cursor:
        products.append(ProductSummary( 
            id=str(doc["_id"]),
            name=doc["name"],
            price=doc["price"]
        ))

    print(products) # [ProductSummary(id='687b71363e94c1afb4710e3f', name='Sneakers', price=2299.0)]
    print(type(products[0]))  # <class 'schemas.ProductSummary'>
    
    # Pagination logic
    page = {
        "next": offset + limit, # This tells the offset value for the next page
        "limit": len(products), # products returned on this page
        "previous": offset - limit if offset - limit >= 0 else -10 # This calculates the offset to go one page back
    }

    return {"data": products, "page": page} # should only includes what's defined in ProductListResponse

# FastAPI (via Pydantic) automatically converts (serializes) Python objects like ProductSummary into JSON before sending the response to the client.
# You don’t need to manually convert Pydantic models to JSON — FastAPI does that for you under the hood.

- The parameters `name`, `size`, `limit`, and `offset` in the `list_products()` function are query parameters. When you send a `GET` request to the `/products` endpoint, these values can be passed through the URL. `/products?name=sneakers`, FastAPI automatically extracts those query parameters and passes them to the `list_products()` function:
    - `name = "sneakers"`

- MongoDB's `.find()` uses logical AND for multiple keys. So both `name` and `sizes.size` must match in the same document.

- `ProductSummary` converts each MongoDB document into a `ProductSummary` object (only `id`, `name`, and `price`) and appended in `products` list.

- `return {"data": products, "page": page}`: It sees `response_model=ProductListResponse`.

**Why are we returning `products` directly but the API response is supposed to be in JSON?**


- Before sending the response:
    - FastAPI checks if the return value matches the schema.
    - It recursively converts all Pydantic model instances into dictionaries using `.dict()` or `.model_dump()` under the hood.
- Then it serializes everything into JSON to return it as an HTTP response.

FastAPI uses Pydantic's `.model_dump()` or equivalent under the hood to serialize the data to JSON format.

**We can do like this also:**

In [ ]:
return ProductListResponse(
    data=products,
    page=page
)

- `offset` means how many records you skip
- `limit` means how many records you fetch
<br>
<hr>

In [ ]:
# Create Order API
@app.post("/orders", response_model=OrderResponse, status_code=201)
async def create_order(order: OrderCreate):
    product_ids = [] # store all valid productId 

    for item in order.items:
        try:
            # item have productId and qty
            product_ids.append(ObjectId(item.productId)) 
        except Exception:
            raise HTTPException(status_code=400, detail=f"Invalid productId: {item.productId}")
            # Exception will raised for invalid id (format), not for id that do not exist in DB

    # Fetch product documents using productId
    products = await product_collection.find({"_id": {"$in": product_ids}}).to_list(length=len(product_ids))

    if len(products) != len(order.items):
        # order.items will contain all items i.e, existing or non existing in DB
        # products will conain only items that exist in DB
        found_ids = {str(p["_id"]) for p in products} # set
        missing_ids = [item.productId for item in order.items if item.productId not in found_ids]
        raise HTTPException(status_code=404, detail=f"Products not found: {missing_ids}")

    # Check stock & prepare deduction plan
    stock_updates = []  # collect (product_id, updated_sizes) to update DB later
    # [(ObjectId("64f6c2..."),[{"size": "S", "quantity": 0}, ...]), ...]

    for item in order.items:
        product = next((p for p in products if str(p["_id"]) == item.productId), None)
        
        if not product:
            continue  # Skip if product is None

        sizes = product.get("sizes", []) # get all sizes of that product
        total_stock = sum(size.get("quantity", 0) for size in sizes) # total stock of that product

        if item.qty > total_stock: # if order placed item qty is greater than total stock of that item
            raise HTTPException(
                status_code=400,
                detail=f"Not enough stock for product '{product['name']}' (available: {total_stock}, requested: {item.qty})"
            )

        # Deduct qty from sizes
        # We are not specifying size during order — we’ll deduct from available sizes in any order, starting with the largest available quantity
        
        qty_to_deduct = item.qty # Keeps track of how much quantity is still left to deduct from available sizes
        
        new_sizes = [] # A new list that will replace the current sizes in the product document after stock is updated.
        # it will store [{"size": value, "quatity": updated_quantity}, ...]
        
        # he order doesn’t specify a size, so deduction happens starting with the first size in sizes.
        for size in sizes:
            available = size["quantity"] # get available quatity of a particular size
            
            if qty_to_deduct == 0: # Check if we already deducted enough
                new_sizes.append(size)
                continue

            deduct = min(qty_to_deduct, available)
            new_sizes.append({
                "size": size["size"],
                "quantity": available - deduct
            })
            qty_to_deduct -= deduct

        stock_updates.append((product["_id"], new_sizes))

    # Apply stock updates in MongoDB
    for product_id, updated_sizes in stock_updates:
        await product_collection.update_one(
            {"_id": product_id},
            {"$set": {"sizes": updated_sizes}}
        )

    # Save order in DB
    order_data = {
        "user_id": order.userId,
        "items": [
            {"product_id": item.productId, "quantity": item.qty}
            for item in order.items
        ]
    }

    result = await order_collection.insert_one(order_data)
    
    return {"id": str(result.inserted_id)}

- `order_ids` will store `productId` in `ObjectId` type.
- `product_collection.find({"_id": {"$in": product_ids}})` will return data in `Cursor` type and we converted into a list.
- `(p for p in products if str(p["_id"]) == item.productId)` is generator.
- `next((p for p in products if str(p["_id"]) == item.productId), None)`: Loop through `products`, as soon as it finds a match, it stops and returns that product. If no matching product is found, it returns `None`.
- `next()` is used to retrieve the next item from an iterator. If there are no more items, it raises `StopIteration` — unless you provide a default value (like `None`), in which case it returns that instead.

**Why are we loop thorugh `order.items` to check valid product but already have all valid product in `products`?** Reason is `order.items` contains the original request from the user, which includes the quantity (`qty`) for each `productId`. Whereas products is just a list of existing product documents from MongoDB. So, you need both the product and the quantity requested for that specific product to do proper stock deduction. That's why.

<br>

**Why are we again loop through `order.items` while saving ordered data in `order_collection` as `order.items` may contain products that does not exist in DB?**

Reason is when code start executing and will reach at this portion:

    if len(products) != len(order.items):
        found_ids = {str(p["_id"]) for p in products}
        missing_ids = [item.productId for item in order.items if item.productId not in found_ids]
        raise HTTPException(status_code=404, detail=f"Products not found: {missing_ids}")

If any product in `order.items` was not found in the database, the request is rejected with HTTP 404 Error. So no invalid/nonexistent product will pass this stage.

So by the time we reach this line `result = await order_collection.insert_one(order_data)`, You can be 100% sure that:

    - All productIds in order.items:
        - Are valid ObjectIds
        - Exist in the database
    - Any invalid or missing products have already triggered an HTTPException

In [ ]:
# Get Orders by User ID
@app.get("/orders/{user_id}", response_model=OrderListResponse)
async def get_user_orders(
    user_id: str = Path(...),
    limit: int = Query(10),
    offset: int = Query(0)
):
    pipeline = [
        {"$match": {"user_id": user_id}},
        {"$sort": {"_id": 1}},
        {"$skip": offset},
        {"$limit": limit},
        {"$unwind": "$items"}, # Splits each order document into multiple documents — one per item in the items array
        {
            "$addFields": { # create new field to join with products
                "items.product_id_obj": { # items.product_id_obj means inside items
                    "$toObjectId": "$items.product_id"
                }
            }
        },
        {
            "$lookup": {
                "from": "products",
                "localField": "items.product_id_obj",
                "foreignField": "_id",
                "as": "product_details"
            }
        },
        {"$unwind": "$product_details"},
        {
            "$group": {
                "_id": "$_id",
                "items": {    # not prrvous items, it is newly created by $group
                    "$push": {
                        "productDetails": {
                            "name": "$product_details.name",
                            "id": {"$toString": "$product_details._id"}
                        },
                        "qty": "$items.quantity"
                    }
                },
                "total": {
                    "$sum": {
                        "$multiply": ["$items.quantity", "$product_details.price"]
                    }
                }
            }
        },
        {
            "$project": {
                "id": {"$toString": "$_id"},
                "items": 1,
                "total": 1
            }
        }
    ]


    orders_cursor = order_collection.aggregate(pipeline)
    orders = [order async for order in orders_cursor]

    # Pagination meta
    page_info = {
        "next": offset + limit,
        "limit": len(orders),
        "previous": offset - limit if offset - limit >= 0 else -10
    }

    return {
        "data": orders,
        "page": page_info
    }

1. `$match`: We are fetching user-specific data from the `orders` collection
→ so that we can filter only those orders which belong to the given `user_id`.

2. `$sort`: We are sorting the user’s orders in ascending order based on `_id`
→ so that we can apply consistent pagination and return results in a predictable order.

3. `$skip`: We are skipping the first `offset` number of documents from the sorted results
→ so that we can support pagination by starting from the correct position.

4. `$limit`: We are limiting the number of documents to `limit`
→ so that we only return a specific number of orders as per pagination settings.
     

In [ ]:
# Output:
{
  "_id": ObjectId("64aa1234567890abcdef1234"),
  "user_id": "64bb1234567890abcdef5678",
  "items": [
    {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2
    },
    {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1
    }
  ]
}


5. `$unwind`: We are deconstructing the `items` array in each order into individual documents
→ so that we can work with each item separately, especially for joining with the `products` collection.   

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1
    }
  }
]


6. `$addFields`: We are converting each `items.product_id` (string) into an ObjectId
→ so that we can match it with `_id` field in the `products` collection during the lookup stage.


In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    }
  }
]


7. `$lookup`: We are joining each item with its full product details from the `products` collection
→ so that we can fetch information like name and price of each product.

In [ ]:
# products collection:
[
  {
    "_id": ObjectId("64cc1234567890abcdef9999"),
    "name": "Keyboard",
    "price": 1000,
    "sizes": [
      {
        "size": "S",
        "quantity": 8
      }
    ]
  },
  {
    "_id": ObjectId("64cc1234567890abcdef8888"),
    "name": "Mouse",
    "price": 500,
    "sizes": [
      {
        "size": "M",
        "quantity": 10
      }
    ]
  }
]


In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    },
    "product_details": [
      {
        "_id": ObjectId("64cc1234567890abcdef9999"),
        "name": "Keyboard",
        "price": 1000,
        "sizes": [
          {
            "size": "S",
            "quantity": 8
          }
        ]
      }
    ]
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    },
    "product_details": [
      {
        "_id": ObjectId("64cc1234567890abcdef8888"),
        "name": "Mouse",
        "price": 500,
        "sizes": [
          {
            "size": "M",
            "quantity": 10
          }
        ]
      }
    ]
  }
]


8. `$unwind`: We are flattening the `product_details` array
→ so that we can easily access individual fields like `product_details.name` and `product_details.price`.

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef9999",
      "quantity": 2,
      "product_id_obj": ObjectId("64cc1234567890abcdef9999")
    },
    "product_details": {
      "_id": ObjectId("64cc1234567890abcdef9999"),
      "name": "Keyboard",
      "price": 1000,
      "sizes": [
        {
          "size": "S",
          "quantity": 8
        }
      ]
    }
  },
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "user_id": "64bb1234567890abcdef5678",
    "items": {
      "product_id": "64cc1234567890abcdef8888",
      "quantity": 1,
      "product_id_obj": ObjectId("64cc1234567890abcdef8888")
    },
    "product_details": {
      "_id": ObjectId("64cc1234567890abcdef8888"),
      "name": "Mouse",
      "price": 500,
      "sizes": [
        {
          "size": "M",
          "quantity": 10
        }
      ]
    }
  }
]


9. `$group`: We are regrouping the split items back into their original order using `_id`
→ so that we can reconstruct the complete order with enriched product info and calculate the total bill.

In [ ]:
# Output:
[
  {
    "_id": ObjectId("64aa1234567890abcdef1234"),
    "items": [
      {
        "productDetails": {
          "name": "Keyboard",
          "id": "64cc1234567890abcdef9999"
        },
        "qty": 2
      },
      {
        "productDetails": {
          "name": "Mouse",
          "id": "64cc1234567890abcdef8888"
        },
        "qty": 1
      }
    ],
    "total": 2500
  }
]


10. `$project`: We are shaping the final output to include the order ID, items list, and total price
→ so that we return a clean and client-friendly response format with only necessary fields.

In [ ]:
# Output:
[
  {
    "id": "64aa1234567890abcdef1234",
    "items": [
      {
        "productDetails": {
          "name": "Keyboard",
          "id": "64cc1234567890abcdef9999"
        },
        "qty": 2
      },
      {
        "productDetails": {
          "name": "Mouse",
          "id": "64cc1234567890abcdef8888"
        },
        "qty": 1
      }
    ],
    "total": 2500
  }
]
